# B2.2 · Threat modelling from what the estate already knows

**Function B — Application Security with an AI SDLC → The AI SDLC: Harnesses and an Agentic AppSec Pipeline**  ·  *Security of AI*

Builds on **[B2.0 · What a harness is, and why you are the one building it](https://spbreed.github.io/cyber-commons/lessons/B2.0.html)**.

| | |
|---|---|
| Tools used | OWASP Threat Dragon, GLM-4.6, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Turn an architecture map into a ranked threat model, then diff it after one entry point is added.

**Why a security engineer needs it.** Threat models are written once, by hand, against a system that has since changed. The control it builds is: stage 5: derive assets, entry points and attack vectors mechanically from the synthesised map.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A threat model produced in a workshop describes the system as it was on the day of the workshop, and it is derived from the code alone — so two deployments of the same repository, one behind a private load balancer with no egress and one on the internet with a wildcard trust policy, get the same model. It is wrong about both.

> **At CyberTravels.** The threat model that said “CyberTravels answers questions” is still on file. Deriving it from the architecture on every release is what would have caught the refund endpoint appearing.

## 2 · The framework

```
   six static inputs, all already in the estate

   code analysis      what the code COULD reach     (stage 4)
   cloud policy       is it on the internet         (security groups, WAF)
   CSPM               is the bucket public TODAY
   entitlements       what the role may do
   IAM                who can become that role
   egress policy      can anything leave
          |
          v  derive, mechanically
   +---------------------------+
   | ranked threats, as data   |
   +---------------------------+
          |
        DIFFED against the last run - and the diff has to count
        ESCALATION, not only arrival, or a terraform-only pull
        request raises every score and passes the gate
```

Stage 5 is the one everybody claims to do and almost nobody re-runs.

A threat model produced in a workshop describes the system as it was on the day
of the workshop. It is stale the moment an entry point is added, and adding an
entry point is a Tuesday. So this stage does not *write* a threat model — it
**derives** one, from evidence the estate already holds, and the useful artefact
is the diff between two runs.

**STRIDE** gives six questions. Against an agentic system each has a shape a
web-application threat model does not:

| STRIDE | In an agentic system |
|---|---|
| **S**poofing | agents share a service account, so "which agent" is unanswerable |
| **T**ampering | untrusted content the agent read becomes an instruction it follows |
| **R**epudiation | the delegation chain is not on the token, so no log answers "on whose behalf" |
| **I**nformation disclosure | an over-broad tool return, or egress that permits anything |
| **D**enial of service | an unbounded loop, or a budget nobody set |
| **E**levation of privilege | a role assumable by `*`, or a scope that included refunds because it included payments |

### Derived from five inputs, not one

The architecture map says what the code *could* reach. It cannot say whether
that path is exposed, what identity walks it, or whether anything can leave at
the end of it — and those three decide whether a finding is a fire.

| Input | What only it can tell you |
|---|---|
| **architecture** | components, flows, sinks, trust levels |
| **CSPM** | that the bucket behind that sink is public *today* |
| **IAM** | who can assume the role, and whether MFA is required |
| **network** | is it internet-facing, and can anything leave |
| **entitlements** | what the identity may do once it is through |

Read only the first and you produce a model that is identical for two
deployments of the same repository — one behind a private load balancer with no
egress, one on the internet with a wildcard trust policy. It is wrong about
both.

This lesson runs the `threat-model-stride` skill. You do not write the code;
you read the procedure, execute it, and read what it produced.

## 3 · The skill

This is `skills/appsec/threat-model-stride/SKILL.md`, verbatim. The frontmatter is what routes a request to it; the body is the procedure a model follows.

In [ ]:
# skills/appsec/threat-model-stride/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: threat-model-stride
description: >-
  Build a STRIDE threat model for an agentic system from the evidence an estate
  already holds — source code, CSPM findings, IAM policies, network policies and
  the tool surface — and emit a ranked threat table plus a trust-boundary
  diagram. Use when asked what could go wrong with an architecture, to
  threat-model an agent, a pipeline or a service, when a threat model needs
  regenerating after an architecture change, or when someone asks which STRIDE
  categories a design actually exposes.
allowed-tools: Read, Grep, Glob, Bash
---

# STRIDE threat modelling for an agentic system

A threat model written in a workshop describes the system as it was on the day
of the workshop. This one is **derived**: every threat traces to a line of
evidence that already exists somewhere in the estate, so regenerating it is
cheap and the useful artefact is the **diff between two runs**.

STRIDE gives six categories. Against an agentic system each one has a specific
shape that a web-application threat model does not:

| STRIDE | In an agentic system |
|---|---|
| **S**poofing | An agent calls downstream as a shared service account, so "which agent" is unanswerable |
| **T**ampering | Untrusted content the agent read becomes an instruction it follows |
| **R**epudiation | The delegation chain is not on the token, so no log answers "on whose behalf" |
| **I**nformation disclosure | An over-broad tool return, or egress that permits anything |
| **D**enial of service | An unbounded loop, or a budget nobody set |
| **E**levation of privilege | A role assumable by `*`, or a scope that includes refunds because it included payments |

## When to use this

Threat-modelling an agent, an MCP server, or a review pipeline; re-running a
model after an architecture change; or answering "which STRIDE categories does
this design actually expose" with something better than an opinion.

## Inputs it expects

Five evidence sources. Any one alone produces a model that is wrong in a
predictable direction — code alone cannot tell you whether a path is exposed,
and CSPM alone cannot tell you what reaches it.

| Input | What only it can tell you |
|---|---|
| `architecture` | components, flows, sinks, trust levels — what *could* be reached |
| `cspm` | live posture findings: what is public *today* |
| `iam` | which roles exist, who may assume them, whether MFA is required |
| `network` | ingress exposure and egress policy — can anything leave |
| `entitlements` | what the running identity may do once it is through |

## Procedure

**1 — Load the five inputs.** Refuse to proceed on fewer. A model built on
`architecture` alone should say so in its output rather than silently scoring as
though the estate were hardened.

**2 — Walk each entry point to each sink.** For every reachable pair, ask the
six STRIDE questions. A category that has no evidence behind it is not a threat;
do not invent one to fill the row.

**3 — Score from evidence, not from feeling.** Base severity comes from the
asset. Then adjust *only* where an input says so: internet-facing, no WAF, a
live CSPM finding on the resource, a role that holds write, a trust policy with
a wildcard, egress open. Record which adjustment fired — the reasons are the
part a reviewer argues with.

**4 — Emit the trust-boundary diagram.** Nodes are components, edges are flows,
and an edge crossing from a lower trust level to a higher one is a boundary.
Boundaries are where findings live; render them differently from ordinary edges.

**5 — Diff against the previous run.** Report new threats *and* escalated ones.
A pull request that changes only infrastructure introduces no new threat and
raises every existing score, so a gate that counts arrivals alone waves it
through.

## Output contract

```json
{
  "inputs_present": ["architecture", "cspm", "iam", "network", "entitlements"],
  "threats": [
    {"id": "T-01", "stride": "S|T|R|I|D|E", "entry": "str", "sink": "str",
     "asset": "str", "score": 0, "reasons": ["internet-facing", "..."],
     "evidence": {"source": "cspm|iam|network|entitlements|architecture",
                  "detail": "str"}}
  ],
  "boundaries": [{"from": "str", "to": "str", "trust": "0->2"}],
  "diagram": "mermaid or svg source",
  "diff": {"new": ["T-07"], "escalated": [{"id": "T-01", "from": 13, "to": 17}]}
}
```

## Failure modes

- **Modelling the code and calling it the system.** The same repository behind
  a private load balancer with default-deny egress and on the internet with a
  wildcard trust policy is two different threat models. Read all five inputs.
- **Counting new threats only.** Escalation is the signal a terraform-only
  change produces, and it is the majority of how an estate gets worse.
- **One row per STRIDE letter.** Six categories is a checklist, not a quota. An
  agent with no state has no meaningful tampering row.
- **Scoring without recording why.** A score nobody can argue with is a score
  nobody will act on.
"""

In [ ]:
# Execute the skill above, using the shared runtime rather than a copy.
import glob, importlib.util, os, sys

# Kaggle mounts an attached kernel under /kaggle/input, and it uses two
# different layouts — /kaggle/input/<slug>/ on some kernels and
# /kaggle/input/notebooks/<user>/<slug>/ on others. Both were observed on the
# same account in the same hour, so match either. The recursive glob is cheap
# here because /kaggle/input holds only what is attached; globbing the working
# tree instead cost eleven seconds a notebook.
_WHERE = (sorted(glob.glob("/kaggle/input/**/cyber-commons-skill-runtime/__script__.py",
                           recursive=True))
          + [os.path.join(p, "skills/_runtime/cyber_commons_skill_runtime.py")
             for p in (".", "..", "../..")])
_found = next((p for p in _WHERE if os.path.isfile(p)), None)
if _found is None:
    # Say what was looked for and what is actually there. "The runtime is
    # missing" on its own costs whoever hits it an afternoon.
    raise SystemExit("The shared skill runtime is missing."
                     "  looked at: " + repr(_WHERE) +
                     "  /kaggle/input holds: " +
                     repr(glob.glob("/kaggle/input/**", recursive=True)[:20]) +
                     "  cwd: " + os.getcwd() +
                     ". On Kaggle it is attached to this notebook as a "
                     "source; locally it is skills/_runtime/ in the repository.")
_spec = importlib.util.spec_from_file_location("cyber_commons_skill_runtime", _found)
cyber_commons_skill_runtime = importlib.util.module_from_spec(_spec)
sys.modules["cyber_commons_skill_runtime"] = cyber_commons_skill_runtime
_spec.loader.exec_module(cyber_commons_skill_runtime)
from cyber_commons_skill_runtime import run_skill

# Split skills/appsec/threat-model-stride/SKILL.md into the two halves an agent uses —
# the frontmatter it routes on, and the body it follows.
meta, body = run_skill(SKILL_MD)

## 4 · Its script

The deterministic half of the skill — the part that has to give the same answer twice so two runs can be diffed. Embedded from `skills/appsec/threat-model-stride/scripts/`.

In [ ]:
# skills/appsec/threat-model-stride/scripts/threat_model.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Derive a STRIDE threat model and a trust-boundary diagram from five inputs.

This is the executable half of the `threat-model-stride` skill. The SKILL.md
next to it is the procedure a model follows; this is the deterministic part it
calls, so the scoring is reproducible and two runs can be diffed.

    python3 threat_model.py     # the synthetic CyberTravels estate, then the
                                # same estate hardened, then the diff

There is no CLI. The module prints its demonstration at import, because it is
embedded into a notebook cell where `__name__` is `__main__` and `sys.argv`
belongs to the kernel — an `if __name__ == "__main__"` block guarded on
arguments ran argparse against a Kaggle kernel's argv and failed the notebook.
Call `model()`, `boundaries()` and `diagram()` directly for the JSON contract.

Standard library only, so it runs on a Kaggle kernel with the internet off.
"""

import ast
import json
from collections import defaultdict

# ---------------------------------------------------------------- the inputs
# Synthetic, and deliberately so: every value below stands in for something a
# real estate already holds, and the point of the lesson is that all five are
# read rather than just the first.

# ------------------------------------------------- the architecture, derived
# The recon stage used to be a lesson of its own and this file retyped its
# result. Retyping is the failure this lesson warns about: a threat model that
# describes the system as somebody once described it, rather than as the code
# is now. So the minimum of the recon stage lives here — parse the sources,
# take the units nothing calls as entry points, take a call into a dangerous
# builtin as a sink — and the model below is built from what that returns.

SOURCES = {
 "src/api/bookings.py": '''
def get_booking(request):
    """HTTP GET /bookings/<ref> - request.args is traveller-controlled."""
    return render(load_booking(request.args["ref"], request.args["owner"]))

def upload_voucher(request):
    """HTTP POST /vouchers - the multipart body is traveller-controlled."""
    return store(request.files["doc"], request.args["name"])

def health(request):
    """HTTP GET /health - no session required."""
    return "ok"
''',
 "src/data/reports.py": '''
def load_booking(ref, owner):
    return DB.execute("SELECT * FROM bookings WHERE ref=" + ref)
''',
 "src/data/docs.py": '''
def store(blob, name):
    open("/srv/vouchers/" + name, "wb").write(blob)
''',
 "src/util/render.py": '''
def render(rows):
    return "\\n".join(str(r) for r in rows)
''',
}

TRUST = {"src/api": 0, "src/util": 1, "src/data": 2}   # 0 = the untrusted edge
DANGEROUS = {"execute": "bookings_db", "open": "voucher_bucket"}
UNAUTHENTICATED = {"health"}          # what the router leaves open
ASSETS = {"bookings_db": {"data": ["customer", "financial"], "value": 5},
          "voucher_bucket": {"data": ["documents"], "value": 3}}


def units_of(src, path):
    """Semantic units and what each one calls. Not lines, not files."""
    out = []
    for fn in [n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef)]:
        calls = sorted({(c.func.id if isinstance(c.func, ast.Name)
                         else getattr(c.func, "attr", ""))
                        for c in ast.walk(fn) if isinstance(c, ast.Call)} - {""})
        out.append({"name": fn.name, "component": path.rsplit("/", 1)[0],
                    "calls": calls})
    return out


def derive(sources):
    """The map: entry points, flows, sinks, and the trust level per component.

    An entry point is a unit nothing in the repository calls — which is what
    makes it reachable from outside. Deriving it beats declaring it, because
    adding a function changes the answer and nobody has to remember to.
    """
    units = [u for p, s in sorted(sources.items()) for u in units_of(s, p)]
    names = {u["name"] for u in units}
    called = {c for u in units for c in u["calls"] if c in names}
    entries = [{"unit": u["name"], "component": u["component"],
                "auth": "none" if u["name"] in UNAUTHENTICATED else "session"}
               for u in units if u["name"] not in called]
    flows = sorted({(u["name"], c) for u in units for c in u["calls"]
                    if c in names or c in DANGEROUS})
    sinks = [{"unit": u["name"], "component": u["component"],
              "resource": DANGEROUS[c], "sink": c}
             for u in units for c in u["calls"] if c in DANGEROUS]
    return {"components": TRUST, "entry_points": sorted(entries, key=lambda e: e["unit"]),
            "flows": flows, "sinks": sorted(sinks, key=lambda s: s["unit"]),
            "assets": ASSETS}


ARCHITECTURE = derive(SOURCES)

CSPM = [
    {"resource": "voucher_bucket", "finding": "bucket policy allows public read",
     "severity": 4},
]

IAM = {"src/api": {"role": "cybertravels-api",
                   "assumable_by": ["ci-deploy-role", "*"],
                   "mfa_required": False}}

NETWORK = {"src/api": {"exposed": "internet", "waf": False,
                       "egress_default_deny": False,
                       "egress_allowed": ["0.0.0.0/0"]}}

ENTITLEMENTS = {"src/api": ["db:select", "db:update",
                            "s3:GetObject", "s3:PutObject"]}

HARDENED = {
    "cspm": [],
    "iam": {"src/api": {"role": "cybertravels-api",
                        "assumable_by": ["ci-deploy-role"], "mfa_required": True}},
    "network": {"src/api": {"exposed": "vpc-only", "waf": True,
                            "egress_default_deny": True,
                            "egress_allowed": ["bookings-db.prod:5432"]}},
    "entitlements": {"src/api": ["db:select", "s3:GetObject"]},
}

# The six questions. Each one applies to a path structurally — an agent that
# calls a sink can always be spoofed, can always repudiate, can always disclose.
# What the evidence changes is the SCORE, not whether the row exists. A model
# that deletes rows when the estate is hardened teaches that hardening removes
# threats; it does not, it removes severity, and the row is what you re-check
# after the next terraform change.
STRIDE = [
    ("S", "Spoofing", "iam",
     lambda c: (2, "the running role is assumable by *")
     if "*" in c["iam"].get("assumable_by", []) else
     (0, "role assumable only by named principals")),
    ("T", "Tampering", "architecture",
     lambda c: (2, "an unauthenticated entry point reaches this sink")
     if c["entry"]["auth"] == "none" else
     (0, "entry point requires a session")),
    ("R", "Repudiation", "iam",
     lambda c: (1, "no MFA on the assumable role, so the actor is not established")
     if not c["iam"].get("mfa_required", False) else
     (0, "MFA required to assume the role")),
    ("I", "Information disclosure", "cspm",
     lambda c: (c["cspm_severity"],
                "a live CSPM finding on the resource this path reaches")
     if c["cspm_severity"] else (0, "no open CSPM finding on the resource")),
    ("D", "Denial of service", "network",
     lambda c: (1, "internet-facing with no WAF in front of it")
     if c["net"].get("exposed") == "internet" and not c["net"].get("waf") else
     (0, "not directly reachable, or a WAF is in front")),
    ("E", "Elevation of privilege", "entitlements",
     lambda c: (1, "the identity holds write, not just read")
     if any(e in c["entitlements"] for e in ("db:update", "s3:PutObject")) else
     (0, "the identity is read-only on this resource")),
]


def reachable(entry, flows):
    adj = defaultdict(list)
    for a, b in flows:
        adj[a].append(b)
    seen, stack = set(), [entry]
    while stack:
        for nxt in adj[stack.pop()]:
            if nxt not in seen:
                seen.add(nxt)
                stack.append(nxt)
    return seen


def model(architecture, cspm, iam, network, entitlements):
    """Step 2 and 3 of the procedure: walk, question, score from evidence."""
    by_resource = defaultdict(int)
    for f in cspm:
        by_resource[f["resource"]] += f["severity"]

    threats, n = [], 0
    for entry in architecture["entry_points"]:
        reach = reachable(entry["unit"], architecture["flows"])
        comp = entry["component"]
        for sink in architecture["sinks"]:
            if sink["unit"] not in reach:
                continue
            asset = architecture["assets"][sink["resource"]]
            ctx = {"entry": entry, "sink": sink, "iam": iam.get(comp, {}),
                   "net": network.get(comp, {}),
                   "entitlements": entitlements.get(comp, []),
                   "cspm_severity": by_resource[sink["resource"]]}
            exposure = 2 if ctx["net"].get("exposed") == "internet" else -3
            egress = 0 if ctx["net"].get("egress_default_deny", True) else 2
            for letter, name, source, assess in STRIDE:
                bump, why = assess(ctx)
                n += 1
                score = max(1, asset["value"] + bump + exposure + egress)
                threats.append({
                    "id": f"T-{n:02d}", "stride": letter, "category": name,
                    "entry": entry["unit"], "sink": sink["unit"],
                    "asset": sink["resource"], "score": score,
                    "mitigated": bump == 0,
                    "reasons": [why]
                               + (["internet-facing"] if exposure > 0 else ["vpc-only"])
                               + (["egress open"] if egress else ["egress default-deny"]),
                    "evidence": {"source": source, "detail": why},
                })
    return sorted(threats, key=lambda t: (-t["score"], t["stride"], t["id"]))


def boundaries(architecture):
    """Step 4: an edge from a lower trust level to a higher one."""
    comp = {}
    for e in architecture["entry_points"]:
        comp[e["unit"]] = e["component"]
    for s in architecture["sinks"]:
        comp[s["unit"]] = s["component"]
    lv = architecture["components"]
    out = []
    for a, b in architecture["flows"]:
        ca, cb = comp.get(a), comp.get(b)
        if ca and cb and lv.get(ca, 0) < lv.get(cb, 0):
            out.append({"from": a, "to": b, "trust": f"{lv[ca]}->{lv[cb]}"})
    return out


def diagram(architecture, bnds):
    """Step 4: mermaid, because it renders in the notebook without a library."""
    lines = ["flowchart LR"]
    for comp, level in sorted(architecture["components"].items()):
        label = comp.replace("/", "_")
        lines.append(f'  subgraph {label}["{comp} · trust {level}"]')
        members = [e["unit"] for e in architecture["entry_points"]
                   if e["component"] == comp]
        members += [s["unit"] for s in architecture["sinks"]
                    if s["component"] == comp]
        for m in sorted(set(members)) or ["·"]:
            lines.append(f"    {m}")
        lines.append("  end")
    crossing = {(b["from"], b["to"]) for b in bnds}
    for a, b in architecture["flows"]:
        lines.append(f"  {a} ==>|BOUNDARY| {b}" if (a, b) in crossing
                     else f"  {a} --> {b}")
    return "\n".join(lines)


def diff(before, after):
    """Step 5: arrivals AND escalations."""
    key = lambda t: (t["stride"], t["entry"], t["sink"])
    prev = {key(t): t["score"] for t in before}
    new = [t["id"] for t in after if key(t) not in prev]
    esc = [{"id": t["id"], "from": prev[key(t)], "to": t["score"]}
           for t in after if key(t) in prev and t["score"] > prev[key(t)]]
    return {"new": new, "escalated": esc}


# ---------------------------------------------------- the demonstration
# What the lesson runs, at module level, so the notebook and a terminal
# both print the same thing. `main()` below is still the CLI, and only
# fires when arguments are given.

threats = model(ARCHITECTURE, cspm=CSPM, iam=IAM, network=NETWORK,
                 entitlements=ENTITLEMENTS)
bnds = boundaries(ARCHITECTURE)

print(f"{'id':6s}{'':2s}{'entry':16s}{'sink':14s}{'score':>6}  why")
print("-" * 92)
for t in threats:
    print(f"{t['id']:6s}{t['stride']:2s}{t['entry']:16s}{t['sink']:14s}"
          f"{t['score']:>6}  {t['reasons'][0]}")

print(f"\n{len(threats)} threats across "
      f"{len({t['stride'] for t in threats})} STRIDE categories")
print(f"{len(bnds)} trust-boundary crossing(s):")
for b in bnds:
    print(f"   {b['from']} -> {b['to']}   ({b['trust']})")
assert len({t["stride"] for t in threats}) == 6

print(diagram(ARCHITECTURE, bnds))

hard = model(ARCHITECTURE, cspm=HARDENED["cspm"], iam=HARDENED["iam"],
              network=HARDENED["network"],
              entitlements=HARDENED["entitlements"])
by_id = {(t["stride"], t["entry"], t["sink"]): t["score"] for t in hard}

print(f"{'':2s}{'entry -> sink':34s}{'deployed':>10}{'hardened':>10}")
print("-" * 58)
for t in threats[:6]:
    k = (t["stride"], t["entry"], t["sink"])
    print(f"{t['stride']:2s}{t['entry'] + ' -> ' + t['sink']:34s}"
          f"{t['score']:>10}{by_id[k]:>10}")

print(f"\nthreat rows: {len(threats)} -> {len(hard)}   (unchanged)")
print(f"max severity: {max(t['score'] for t in threats)} -> "
      f"{max(t['score'] for t in hard)}")
print()
print("The rows do not disappear. Hardening removes severity, not threats -")
print("and the row is what you re-check after the next terraform change.")
assert len(hard) == len(threats)
assert max(t["score"] for t in hard) < max(t["score"] for t in threats)

## 5 · Execute it against CyberTravels

Five synthetic inputs, standing in for what a real estate already holds.

## 6 · The diagram it emits

Mermaid, so it renders here and on the lesson page without a library. Double arrows are trust-boundary crossings — the edges every finding turned out to live on.

## 7 · The same code, a hardened estate

Not one line of CyberTravels' source changes. Only the four evidence inputs around it do.

## What you just proved

The skill loads with its routing description and procedure, then derives twelve threats across all six STRIDE categories from five synthetic inputs, each carrying the evidence line that set its score. It emits a mermaid diagram marking the two trust-boundary crossings. Re-running against a hardened estate — same code, four different evidence inputs — keeps every row and drops the maximum severity from 11 to 1.

## Your turn

Point the skill at one of your own services. The work is not the model, it is collecting the five inputs: if any of them is "in somebody's head", that is the input your threat model is currently guessing at, and the guess is always the optimistic one.

---

**Next → [B2.3 · Vulnerability auditing: three generations of SAST](https://spbreed.github.io/cyber-commons/lessons/B2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*